In [1]:
import json, math, shutil
from pathlib import Path

import cv2
import matplotlib
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.auto import tqdm

# ── font (English only in plots) ──
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.unicode_minus': False,
    'figure.dpi': 100,
})

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

CAPTURES_DIR = Path('/home/vsc/LLM_TUNE/aic_sejong/aic_data/captures')
VIEWS = ['left', 'center', 'right']

ALL_SESSIONS = sorted([
    p for p in CAPTURES_DIR.iterdir()
    if p.is_dir() and (p / 'steps.jsonl').exists()
])

def parse_session(d: Path) -> dict:
    parts = d.name.split('_')
    return {'session': d.name, 'task_type': parts[2], 'rail': parts[3], 'session_dir': d}

print(f'{len(ALL_SESSIONS)} valid sessions')

ModuleNotFoundError: No module named 'cv2'

## Section 1-1 — 데이터 파싱 (steps.jsonl)

### 무엇을 보는가
각 에피소드 디렉토리의 `steps.jsonl`을 읽어 하나의 DataFrame으로 합칩니다.  
행 1개 = 스텝 1개이며, 이후 모든 분석의 입력이 됩니다.

### 어떻게 판단하는가
- **Total steps 수**: 세션 수 × 평균 스텝 수가 예상 범위인지 확인.  
  스텝이 비정상적으로 적은 세션은 도중 종료된 에피소드일 가능성이 있습니다.
- **컬럼 결측 확인**: `tip_x_error`, `plug_x` 등 핵심 컬럼에 NaN이 많으면  
  해당 필드가 기록되지 않은 세션이 섞인 것이므로 필터링이 필요합니다.


In [ ]:

# ── 1-1: Parse steps.jsonl ───────────────────────────────────────────────────
step_rows = []
for sess_dir in tqdm(ALL_SESSIONS, desc='Parsing steps'):
    info = parse_session(sess_dir)
    with open(sess_dir / 'steps.jsonl') as f:
        for line in f:
            d = json.loads(line)
            obs   = d.get('observation', {})
            ext   = d.get('extras', {})
            tfm   = d.get('transforms', {})
            wrench = obs.get('wrist_wrench', {})
            force  = wrench.get('force', {})
            torque = wrench.get('torque', {})
            ctrl   = obs.get('controller_state', {})
            tcp    = ctrl.get('tcp_error', [None]*6)
            plug_t = tfm.get('plug', {}).get('translation', {})
            port_t = tfm.get('port', {}).get('translation', {})
            step_rows.append({
                'session':    info['session'],
                'task_type':  info['task_type'],
                'rail':       info['rail'],
                'step':       d['step'],
                'phase':      d['phase'],
                'tip_x_error': ext.get('tip_x_error'),
                'tip_y_error': ext.get('tip_y_error'),
                'z_offset':   ext.get('z_offset'),
                'force_x': force.get('x'), 'force_y': force.get('y'), 'force_z': force.get('z'),
                'torque_x': torque.get('x'), 'torque_y': torque.get('y'), 'torque_z': torque.get('z'),
                'plug_x': plug_t.get('x'), 'plug_y': plug_t.get('y'), 'plug_z': plug_t.get('z'),
                'port_x': port_t.get('x'), 'port_y': port_t.get('y'), 'port_z': port_t.get('z'),
            })

steps_df = pd.DataFrame(step_rows)
print(f'Total steps: {len(steps_df):,}')


## Section 1-2 — Phase & Task 분포 불균형

### 무엇을 보는가
전체 스텝 수를 phase(approach / insert / stabilize)와 task_type(nic / sc)으로 나누어  
각 카테고리가 데이터셋에서 차지하는 비율을 시각화합니다.

### 어떻게 판단하는가
| 확인 항목 | 판단 기준 | 대응 |
|-----------|----------|------|
| insert 비율 | > 60% → 인코더가 insert 외관에 과도하게 특화됨 | approach 오버샘플링 |
| nic : sc 비율 | 70:30 이상 치우침 → 배치에서 특정 태스크 지배 | 배치 내 1:1 샘플링 |
| stabilize 비율 | 매우 낮음 → 학습 시 무시해도 됨 | 별도 처리 불필요 |

**핵심 메시지**: 불균형한 phase/task 분포는 인코더가 "삽입 직전 자세"에만 최적화되고  
approach 구간(로봇이 이동하는 다양한 시각 정보)을 제대로 학습하지 못하는 원인이 됩니다.


In [ ]:

# ── 1-2: Phase & Task imbalance ──────────────────────────────────────────────
phase_cnt  = steps_df['phase'].value_counts().reindex(['approach','insert','stabilize'])
task_cnt   = steps_df['task_type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

phase_cnt.plot(kind='bar', ax=axes[0], color=['#4C72B0','#DD8452','#55A868'], rot=0)
for bar, v in zip(axes[0].patches, phase_cnt):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
                 f'{v/phase_cnt.sum()*100:.1f}%', ha='center', fontsize=9)
axes[0].set_title('Phase Distribution (step count)')
axes[0].set_ylabel('Steps')

task_cnt.plot(kind='bar', ax=axes[1], color=['#4C72B0','#DD8452'], rot=0)
for bar, v in zip(axes[1].patches, task_cnt):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
                 f'{v/task_cnt.sum()*100:.1f}%', ha='center', fontsize=9)
axes[1].set_title('Task Type Distribution (step count)')
axes[1].set_ylabel('Steps')

plt.suptitle('Dataset Imbalance  [Input Bias Source]', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n[Bias] insert dominates: {:.0f}% of all frames'.format(
    phase_cnt['insert']/phase_cnt.sum()*100))
print('[Bias] nic dominates: {:.0f}% of all frames'.format(
    task_cnt['nic']/task_cnt.sum()*100))


## Section 2-1 — Tip Error 분포 (Output Space Bias)

### 무엇을 보는가
`tip_x_error` / `tip_y_error` = 포트 위치 - 플러그 위치 (월드 프레임, 미터 단위).  
이 값이 0에 수렴하면 삽입 성공입니다. 분포가 얼마나 0 중심인지, 대칭인지를 봅니다.

### 어떻게 판단하는가
**2D scatter (approach / insert 위상별)**
- approach: 퍼진 구름 형태 정상. 너무 좁으면 시작 자세 다양성이 없음.
- insert: 원점으로 수렴하는 나선 패턴이 정상. 수렴하지 않으면 제어기 실패 에피소드 포함.

**평균 bias (mean_x, mean_y)**
| 조건 | 의미 | 대응 |
|------|------|------|
| `|mean| < std × 0.3` | 오차 분포가 대칭 → 출력 편향 없음 (OK) | 별도 처리 불필요 |
| `|mean| > std × 0.3` | 특정 방향으로 치우침 → 출력 공간 편향 (BIASED) | 삽입 시작 시 Gaussian XY offset 추가로 다양화 |

**수렴 그래프**: insert step 진행에 따라 `|tip_error|`가 단조 감소하지 않으면  
진동하는 에피소드가 섞인 것이므로 episode-level 필터링을 고려하세요.


In [ ]:

# ── 2-1: tip_error 2D scatter + convergence over step ────────────────────────
# tip_x/y_error = port.xy - plug.xy  (world frame, metres)
# I-controller converges these to 0 → distribution biased toward 0

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) 2D scatter: approach
for ax, phase, title in zip(
    axes,
    ['approach', 'insert', 'insert'],
    ['Approach: tip XY error', 'Insert: tip XY error', 'Insert: error convergence']
):
    if title.endswith('convergence'):
        for sess, sg in steps_df[steps_df['phase']=='insert'].groupby('session'):
            err_mag = np.sqrt(sg['tip_x_error']**2 + sg['tip_y_error']**2) * 1000
            ax.plot(sg['step'] - sg['step'].min(), err_mag,
                    alpha=0.3, lw=0.7, color='steelblue')
        ax.set_xlabel('Step (within insert phase)')
        ax.set_ylabel('|tip_error| (mm)')
        ax.set_title(title)
    else:
        sub = steps_df[steps_df['phase']==phase]
        ax.scatter(sub['tip_x_error']*1000, sub['tip_y_error']*1000,
                   alpha=0.05, s=2, c=sub['step'], cmap='viridis')
        ax.axhline(0, color='red', lw=0.5)
        ax.axvline(0, color='red', lw=0.5)
        ax.set_xlabel('tip_x_error (mm)')
        ax.set_ylabel('tip_y_error (mm)')
        ax.set_title(title)
        ax.set_aspect('equal')

plt.suptitle('Output Bias: tip error distribution  [Output Space]', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

ins = steps_df[steps_df['phase']=='insert']
print('\ntip_error stats (insert phase):')
display(ins.groupby('task_type')[['tip_x_error','tip_y_error']].describe().round(5))


## Section 2-2 — z_offset 궤적 & force_z 분포

### 무엇을 보는가
- `z_offset`: 플러그가 포트 삽입 방향으로 얼마나 이동했는지 (미터).
- `force_z`: 삽입 방향 접촉력 (뉴턴). 접촉 시 급격히 증가.

### 어떻게 판단하는가
**z_offset 궤적**
- 모든 세션이 비슷한 궤적 → z_offset 다양성 없음 → 인코더가 깊이를 암기할 수 있음.
- 세션마다 다른 형태 → 다양한 삽입 전략 존재 → 좋음.
- 완전히 평탄한 궤적(z_offset 변화 없음) → 삽입 실패 에피소드 의심.

**force_z by phase**
| 패턴 | 의미 |
|------|------|
| approach: force_z ≈ 0 | 접촉 없음 → 정상 |
| insert: force_z 분포 넓음 | 다양한 접촉력 → 데이터 다양성 좋음 |
| insert: force_z 항상 0 | 접촉 감지 실패 또는 F/T 센서 Tare 오류 |
| force_z < 0 (당기는 힘) | 케이블 장력 또는 역삽입 → 필터링 검토 |


In [ ]:

# ── 2-2: z_offset trajectory & force_z distribution ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# z_offset per session
for task, ax in zip(['nic','sc'], axes[:2]):
    for sess, sg in steps_df[steps_df['task_type']==task].groupby('session'):
        ax.plot(sg['step'], sg['z_offset'], alpha=0.35, lw=0.8)
    ax.set_xlabel('Step')
    ax.set_ylabel('z_offset (m)')
    ax.set_title(f'z_offset trajectory  [{task.upper()}]')

# force_z by phase
sns.histplot(data=steps_df, x='force_z', hue='phase', bins=50,
             stat='density', common_norm=False, element='step', fill=False, ax=axes[2])
axes[2].set_title('force_z distribution by phase')
axes[2].set_xlabel('force_z (N)')

plt.suptitle('Output Bias: trajectory & contact force  [Output Space]', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## Section 3-1 — 이미지 샘플링 (분석 준비)

### 무엇을 보는가
각 세션에서 균등 간격으로 N개의 프레임 경로를 샘플링하여 `img_df`를 구성합니다.  
이후 3-2~4-5의 모든 이미지 분석에서 이 테이블을 입력으로 사용합니다.

### 어떻게 판단하는가
- `Sampled image records` 수가 예상값 (`세션수 × N_PER_SESSION × 3 뷰`)과 맞는지 확인.
- 적은 경우 → 일부 뷰 이미지 파일이 누락된 세션 존재 → `path` 컬럼에서 빠진 세션 특정 후 제거.


In [ ]:

# ── 3-1: Image sampling ──────────────────────────────────────────────────────
RESIZE_H, RESIZE_W = 256, 288
N_PER_SESSION = 30

def load_gray(path, h=RESIZE_H, w=RESIZE_W):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    return cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA).astype(np.float32)

def load_rgb(path, h=RESIZE_H, w=RESIZE_W):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA)

# build image path table
img_rows = []
for sess_dir in ALL_SESSIONS:
    info = parse_session(sess_dir)
    with open(sess_dir / 'steps.jsonl') as f:
        lines = f.readlines()
    idxs = np.linspace(0, len(lines)-1, N_PER_SESSION, dtype=int)
    for i in idxs:
        d = json.loads(lines[i])
        obs = d.get('observation', {})
        for v in VIEWS:
            k = f'{v}_image'
            if k in obs:
                img_rows.append({
                    'session': info['session'], 'task_type': info['task_type'],
                    'rail': info['rail'], 'view': v, 'phase': d['phase'],
                    'step': d['step'], 'path': Path(obs[k]['path']),
                })

img_df = pd.DataFrame(img_rows)
print(f'Sampled image records: {len(img_df):,}')


## Section 3-2 — 픽셀 분산 히트맵 (Input Spatial Bias)

### 무엇을 보는가
여러 프레임에 걸쳐 각 픽셀 위치의 분산을 계산합니다.  
분산이 높은 픽셀 = 로봇/플러그가 자주 지나가는 위치.  
분산이 낮은 픽셀 = 항상 같은 배경 → 인코더가 무시해도 되는 영역.

### 어떻게 판단하는가
**분산 에너지 집중도 (spatial concentration)**

| 값 | 의미 | 대응 |
|----|------|------|
| < 0.15 (BIASED) | 분산의 50%가 전체 픽셀의 15% 미만에 집중 → 공간 편향 심각 | RandomCrop + AffineTransform 필수 |
| 0.15 ~ 0.30 (caution) | 중간 수준 집중 | 약한 augmentation 권장 |
| > 0.30 (OK) | 분산이 이미지 전반에 고루 분포 | augmentation 선택적 |

**peak 좌표 (cyan + 표시)**: 가장 많이 움직이는 위치가 이미지 중앙 근처인지 확인.  
치우쳐 있으면 해당 뷰의 카메라 각도를 재검토하거나 crop 범위를 조정하세요.


In [ ]:

# ── 3-2: Spatial variance heatmap — where does motion concentrate? ────────────
# High variance pixel = robot/plug passes through that location frequently
# If concentrated → spatial input bias

var_maps = {}
for (view, task), sub in tqdm(img_df.groupby(['view','task_type']), desc='Variance maps'):
    frames = []
    for path in sub['path']:
        try: frames.append(load_gray(path))
        except: continue
    if len(frames) > 1:
        var_maps[(view, task)] = np.stack(frames).var(axis=0)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for row_i, task in enumerate(['nic','sc']):
    for col_i, view in enumerate(VIEWS):
        ax = axes[row_i][col_i]
        vm = var_maps.get((view, task))
        if vm is None: continue
        im = ax.imshow(vm, cmap='hot')
        plt.colorbar(im, ax=ax, fraction=0.046)
        hy, hx = np.unravel_index(vm.argmax(), vm.shape)
        ax.scatter([hx],[hy], c='cyan', s=60, marker='+', linewidths=2)
        ax.set_title(f'{task.upper()} — {view}  peak=({hx},{hy})')
        ax.axis('off')

plt.suptitle('Pixel Variance Heatmap  [Input Spatial Bias]', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Spatial concentration: what fraction of pixels hold 50% of variance energy?
def spatial_conc(vm, thr=0.5):
    flat = np.sort(vm.flatten())[::-1]
    cs = np.cumsum(flat)
    n = np.searchsorted(cs, cs[-1]*thr) + 1
    return n / flat.size

print('\nSpatial concentration (fraction of pixels holding 50% of variance energy):')
print(f'  {"view":8s} {"task":5s}  value   note')
for (view, task), vm in sorted(var_maps.items()):
    c = spatial_conc(vm)
    flag = 'BIASED' if c < 0.15 else ('OK' if c > 0.30 else 'caution')
    print(f'  {view:8s} {task:5s}  {c:.3f}   {flag}')


## Section 3-3 — 8×8 Grid 분산 (공간 활성 영역 정량화)

### 무엇을 보는가
이미지를 8×8 격자로 나누어 각 셀의 평균 분산을 normalized 값으로 표시합니다.  
3-2의 픽셀 단위 분석을 '어느 구역이 중요한가'로 요약한 버전입니다.

### 어떻게 판단하는가
- **활성 셀 수**: 값 > 0.5인 셀이 64개 중 몇 개인지.  
  10개 미만이면 이미지의 85% 이상이 정적 배경 → crop하거나 ROI masking 검토.
- **NIC vs SC 비교**: 태스크별로 활성 영역이 다르면 뷰별·태스크별 crop 범위를 따로 설정하는 게 유리합니다.
- **뷰별 비교**: center 뷰가 가장 정보량이 많아야 정상. left/right가 더 집중되어 있다면 카메라 배치 이슈 가능성.


In [ ]:

# ── 3-3: 8x8 grid variance heatmap — which grid cells are active? ─────────────
GRID = 8
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

for row_i, task in enumerate(['nic','sc']):
    for col_i, view in enumerate(VIEWS):
        ax = axes[row_i][col_i]
        vm = var_maps.get((view, task))
        if vm is None: continue
        H, W = vm.shape
        gh, gw = H//GRID, W//GRID
        grid = np.array([[vm[i*gh:(i+1)*gh, j*gw:(j+1)*gw].mean()
                          for j in range(GRID)] for i in range(GRID)])
        grid /= grid.max() + 1e-8
        sns.heatmap(grid, ax=ax, cmap='YlOrRd', annot=True, fmt='.2f',
                    cbar=False, linewidths=0.3, vmin=0, vmax=1)
        ax.set_title(f'{task.upper()} — {view}')

plt.suptitle(f'{GRID}x{GRID} Grid Variance  [Input Spatial Bias]', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## Section 4-1 — CV 특징 함수 정의

이후 4-2~4-5에서 사용되는 이미지 특징 추출 함수들을 정의합니다.

| 함수 | 측정 대상 | 의미 |
|------|----------|------|
| `fft_high_ratio` | 고주파 에너지 비율 | 높을수록 엣지/텍스처 풍부 |
| `hue_entropy` | 색상 다양성 (엔트로피) | 높을수록 다채로운 색상 분포 |
| `fg_bbox` | 전경 blob 중심 좌표 | 물체가 이미지 어느 위치에 있는가 |

> **참고**: 이 셀은 함수 정의만 합니다. 실행 결과는 4-2부터 나타납니다.


In [ ]:

# ── 4-1: CV feature functions ────────────────────────────────────────────────
def fft_high_ratio(gray, frac=0.18):
    f = np.fft.fftshift(np.fft.fft2(gray.astype(np.float32)))
    mag = np.abs(f)
    h, w = gray.shape
    cy, cx = h//2, w//2
    ry, rx = int(h*frac/2), int(w*frac/2)
    mask = np.zeros_like(gray, dtype=bool)
    mask[cy-ry:cy+ry+1, cx-rx:cx+rx+1] = True
    return float(mag[~mask].sum() / (mag.sum()+1e-6))

def hue_entropy(hsv, bins=36):
    sat = hsv[:,:,1]
    hue = hsv[:,:,0]
    valid = sat > 25
    if valid.sum() < 64: return np.nan
    hist, _ = np.histogram(hue[valid], bins=bins, range=(0,180), density=True)
    hist = hist[hist>0]
    return float(-(hist * np.log(hist+1e-9)).sum())

def fg_bbox(rgb):
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    h, w = lab.shape[:2]
    border = np.concatenate([lab[:16].reshape(-1,3), lab[-16:].reshape(-1,3),
                              lab[:,:16].reshape(-1,3), lab[:,-16:].reshape(-1,3)])
    dist = np.linalg.norm(lab.astype(np.float32) - np.median(border,0).astype(np.float32), axis=2)
    mask = ((dist > np.percentile(dist,82))*255).astype(np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  np.ones((5,5),np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((9,9),np.uint8))
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask)
    ic = np.array([w/2, h/2])
    best, bscore = None, None
    for i in range(1, n):
        a = stats[i, cv2.CC_STAT_AREA]
        if a < 80: continue
        cx = stats[i, cv2.CC_STAT_LEFT] + stats[i, cv2.CC_STAT_WIDTH]/2
        cy = stats[i, cv2.CC_STAT_TOP]  + stats[i, cv2.CC_STAT_HEIGHT]/2
        s = a - 3000*np.linalg.norm((np.array([cx,cy])-ic)/np.array([w,h]))
        if bscore is None or s > bscore: bscore, best = s, i
    if best is None: return np.nan, np.nan
    s = stats[best]
    return (s[cv2.CC_STAT_LEFT]+s[cv2.CC_STAT_WIDTH]/2)/w, \
           (s[cv2.CC_STAT_TOP] +s[cv2.CC_STAT_HEIGHT]/2)/h

print('CV functions ready')


## Section 4-2 — CV 특징 추출

### 무엇을 보는가
세션별로 approach / insert 위상의 프레임을 샘플링하여  
laplacian_var, noise_std, fft_high, edge_density, brightness, saturation, hue_entropy, fg_cx/cy  
를 계산하고 `cv_df`에 저장합니다.

### 어떻게 판단하는가
- `CV records` 수가 `세션수 × N_PER_PHASE × 2(위상) × 3(뷰)` 에 근접하면 정상.
- 크게 적으면 이미지 읽기 오류 세션 존재 → try/except가 처리했지만 로그 확인 권장.


In [ ]:

# ── 4-2: CV feature extraction (view x task_type x phase) ────────────────────
N_PER_PHASE = 12
cv_rows = []

for sess_dir in tqdm(ALL_SESSIONS, desc='CV features'):
    info = parse_session(sess_dir)
    with open(sess_dir / 'steps.jsonl') as f:
        all_lines = f.readlines()
    phase_groups = {'approach':[], 'insert':[]}
    for ln in all_lines:
        d = json.loads(ln)
        if d['phase'] in phase_groups:
            phase_groups[d['phase']].append(d)

    for ph, records in phase_groups.items():
        step = max(1, len(records)//N_PER_PHASE)
        for d in records[::step][:N_PER_PHASE]:
            obs = d.get('observation', {})
            for view in VIEWS:
                key = f'{view}_image'
                if key not in obs: continue
                try:
                    rgb  = load_rgb(Path(obs[key]['path']))
                    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
                    gf   = gray.astype(np.float32)
                    hsv  = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
                    val, sat = hsv[:,:,2], hsv[:,:,1]

                    lap  = float(cv2.Laplacian(gf, cv2.CV_32F).var())
                    ns   = float((gf - cv2.GaussianBlur(gf,(0,0),1.0)).std())
                    fft  = fft_high_ratio(gray)
                    edge = float((cv2.Canny(gray,40,120)>0).mean())
                    bri  = float(val.mean()/255)
                    sat_m= float(sat.mean()/255)
                    hent = hue_entropy(hsv)
                    fx, fy = fg_bbox(rgb)

                    cv_rows.append({
                        'session': info['session'], 'task_type': info['task_type'],
                        'phase': ph, 'view': view,
                        'laplacian_var': lap, 'noise_std': ns,
                        'fft_high': fft, 'edge_density': edge,
                        'brightness': bri, 'saturation': sat_m, 'hue_entropy': hent,
                        'fg_cx': fx, 'fg_cy': fy,
                    })
                except Exception: continue

cv_df = pd.DataFrame(cv_rows)
print(f'CV records: {len(cv_df):,}')


## Section 4-3 — 노이즈 / 텍스처 / 주파수 (Input Feature Bias)

### 무엇을 보는가
laplacian_var (선명도), noise_std (고주파 노이즈), fft_high_ratio (주파수 분포),  
edge_density (엣지 밀도)를 뷰 × task_type × phase 조합으로 boxplot 비교합니다.

### 어떻게 판단하는가
| 특징 | 확인 포인트 | 이상 신호 |
|------|-----------|----------|
| laplacian_var | approach vs insert 차이 | insert가 극도로 낮음 → 흐릿한 근접 촬영 (정상 범위) |
| noise_std | 뷰 간 차이 | 특정 뷰만 높음 → 카메라 센서 노이즈 차이 |
| fft_high_ratio | task 간 차이 | NIC ≠ SC 이면 텍스처 복잡도 다름 → 인코더 용량 설정에 반영 |
| edge_density | phase 간 차이 | approach > insert 이면 정상 (가까울수록 엣지 감소) |

특징값 분포가 phase별로 크게 다르면 → phase-conditional augmentation 또는 별도 헤드 고려.


In [ ]:

# ── 4-3: Noise / Texture / Frequency  (view x phase) ─────────────────────────
cv_df['view_task'] = cv_df['view'] + '/' + cv_df['task_type']
order = [f'{v}/{t}' for v in VIEWS for t in ['nic','sc']]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.flat,
                   ['laplacian_var','noise_std','fft_high','edge_density']):
    sns.boxplot(data=cv_df, x='view_task', y=col, hue='phase',
                order=order, ax=ax, showfliers=False)
    ax.set_title(col)
    ax.set_xlabel('')
    ax.tick_params(axis='x', labelsize=8)
    if ax != axes.flat[0]:
        legend = ax.get_legend()
        if legend: legend.remove()

handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right', title='phase')
plt.suptitle('Noise / Texture / Frequency  [Input Feature Bias]', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## Section 4-4 — HSV 스칼라 특징 (Input Appearance Bias)

### 무엇을 보는가
brightness (명도), saturation (채도), hue_entropy (색상 다양성)를 뷰 × task_type으로 비교하고,  
추가로 approach vs insert 간 brightness 분포 차이를 히스토그램으로 시각화합니다.

### 어떻게 판단하는가
**뷰 간 비교**
- brightness: 세 뷰가 비슷하면 조명 균일 → 좋음. 특정 뷰만 극도로 어두우면 카메라 각도 문제.
- hue_entropy: 낮으면(< 1.5) 색상이 단조로움 → ColorJitter augmentation 필수.

**Phase-domain Shift (하단 히스토그램)**
| 패턴 | 의미 | 대응 |
|------|------|------|
| approach / insert 분포 완전히 겹침 | 위상 변화가 밝기에 영향 없음 → 좋음 | 별도 처리 불필요 |
| approach가 전반적으로 밝음 | 먼 거리 → 배경 비율 높음 | 정상 (거리 감소로 배경 가려짐) |
| insert가 더 밝음 | 근접 시 조명 반사 → 포화 위험 | Brightness 상한 clipping 고려 |


In [ ]:

# ── 4-4: HSV scalar features  (view x task) ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['brightness','saturation','hue_entropy']):
    sns.boxplot(data=cv_df, x='view', y=col, hue='task_type', ax=ax, showfliers=False)
    ax.set_title(col)

plt.suptitle('HSV Scalar Features  [Input Appearance Bias]', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# brightness distribution by phase (key: are approach/insert frames visually different?)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, view in zip(axes, VIEWS):
    sns.histplot(data=cv_df[cv_df['view']==view], x='brightness', hue='phase',
                 bins=25, stat='density', common_norm=False,
                 element='step', fill=False, ax=ax)
    ax.set_title(f'{view}  brightness by phase')
plt.suptitle('Brightness Distribution: approach vs insert  [Phase-domain Shift]', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## Section 4-5 — 전경 Blob 중심 분포 (Input Spatial Bias)

### 무엇을 보는가
각 프레임에서 배경과 다른 '움직이는 물체' (로봇 그리퍼, 플러그, 케이블)의 중심 좌표를  
normalized (0~1) 좌표로 scatter plot하여 뷰 × task_type 별로 비교합니다.

### 어떻게 판단하는가
**IQR (사분위 범위) 기준**
| cx_IQR & cy_IQR | 의미 | 대응 |
|-----------------|------|------|
| 둘 다 < 0.10 (BIASED) | 물체가 항상 같은 위치에만 등장 → 공간 편향 심각 | RandomAffine / RandomCrop 필수 |
| 0.10 ~ 0.20 | 중간 다양성 | 약한 augmentation 권장 |
| > 0.20 | 물체가 다양한 위치에 등장 → 공간 다양성 충분 | augmentation 선택적 |

**NIC vs SC 겹침 여부**: 두 태스크의 점군이 겹치면 인코더가 위치만으로 태스크를 구별할 수 없음.  
분리되어 있으면 공간적 위치가 태스크 구별에 유용한 단서가 됩니다.


In [ ]:

# ── 4-5: Foreground bbox center distribution  [Input Spatial Bias] ──────────
fg = cv_df.dropna(subset=['fg_cx','fg_cy'])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, view in zip(axes, VIEWS):
    sub = fg[fg['view']==view]
    for task, color, m in [('nic','#2196F3','o'),('sc','#FF5722','^')]:
        ts = sub[sub['task_type']==task]
        ax.scatter(ts['fg_cx'], ts['fg_cy'], alpha=0.3, s=8,
                   c=color, marker=m, label=task)
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.invert_yaxis()
    ax.axhline(0.5, color='gray', lw=0.5, ls='--')
    ax.axvline(0.5, color='gray', lw=0.5, ls='--')
    ax.set_title(f'{view}  foreground center')
    ax.set_xlabel('cx (norm)'); ax.set_ylabel('cy (norm)')
    ax.legend(markerscale=3)

plt.suptitle('Foreground Blob Center Distribution  [Input Spatial Bias]', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nForeground IQR (lower = more concentrated = more biased):')
print(f'  {"view":8s} {"task":5s}  cx_IQR  cy_IQR')
for (view, task), sub in fg.groupby(['view','task_type']):
    cx_iqr = sub['fg_cx'].quantile(0.75) - sub['fg_cx'].quantile(0.25)
    cy_iqr = sub['fg_cy'].quantile(0.75) - sub['fg_cy'].quantile(0.25)
    flag = 'BIASED' if cx_iqr < 0.10 and cy_iqr < 0.10 else ''
    print(f'  {view:8s} {task:5s}  {cx_iqr:.3f}   {cy_iqr:.3f}  {flag}')


## Section 5-1 — Cross-view 픽셀 차이 (View Diversity)

### 무엇을 보는가
동일 시점(same step)에서 left / center / right 뷰를 pair별로 픽셀 차이를 구하고  
그 평균을 히트맵으로 시각화합니다.

### 어떻게 판단하는가
| mean diff | 의미 | 인코더 설계 시사점 |
|-----------|------|-------------------|
| > 20 (0~255 scale) | 뷰 간 시각 정보 차이가 큼 | 뷰별 독립 인코더 또는 뷰 ID 임베딩 필요 |
| 10 ~ 20 | 중간 다양성 | 공유 인코더 + 뷰별 어댑터 고려 |
| < 10 | 뷰 간 차이 거의 없음 | 단일 공유 인코더로 충분 |

**히트맵 패턴**: 차이가 집중된 영역이 로봇 관절/그리퍼 위치와 일치하면 정상.  
배경에서 차이가 크면 카메라 화이트밸런스 또는 노출 설정이 다른 것이므로 전처리 시 normalize 필요.


In [ ]:

# ── 5-1: Cross-view pixel difference  (same frame, insert phase) ─────────────
diff_lc = np.zeros((RESIZE_H, RESIZE_W))
diff_lr = np.zeros((RESIZE_H, RESIZE_W))
n = 0
for sess_dir in tqdm(ALL_SESSIONS[:10], desc='Pixel diff'):
    with open(sess_dir / 'steps.jsonl') as f:
        lines = f.readlines()
    ins = [l for l in lines if json.loads(l)['phase']=='insert']
    for ln in ins[::max(1,len(ins)//10)][:10]:
        d = json.loads(ln)
        obs = d.get('observation', {})
        paths = {v: obs.get(f'{v}_image',{}).get('path') for v in VIEWS}
        if not all(paths.values()): continue
        try:
            lg = load_gray(Path(paths['left']))
            cg = load_gray(Path(paths['center']))
            rg = load_gray(Path(paths['right']))
            diff_lc += np.abs(lg - cg)
            diff_lr += np.abs(lg - rg)
            n += 1
        except: continue

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, dm, title in [(axes[0], diff_lc/n, 'left vs center'),
                      (axes[1], diff_lr/n, 'left vs right')]:
    im = ax.imshow(dm, cmap='hot')
    plt.colorbar(im, ax=ax)
    ax.set_title(f'{title}  mean diff = {dm.mean():.2f}')
    ax.axis('off')

plt.suptitle('Cross-view Pixel Difference  [View Diversity]', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Frames used: {n}')


## Section 5-2 — Cross-view 특징 상관관계 (Shared Encoder Validity)

### 무엇을 보는가
같은 스텝에서 left / center / right 뷰의 brightness, laplacian, fft, edge 값을  
pairwise Pearson r로 비교합니다. 뷰 간 상관이 높으면 같은 장면을 보고 있다는 의미입니다.

### 어떻게 판단하는가
| Pearson r | 의미 | 인코더 설계 결론 |
|-----------|------|----------------|
| > 0.80 (모든 특징) | 뷰들이 동일한 시각 정보 공유 | **단일 공유 인코더 사용 가능** ✅ |
| 0.50 ~ 0.80 | 부분적 공유 | 공유 인코더 + 뷰별 projection head |
| < 0.50 (일부 특징) | 특정 특징에서 뷰 간 정보 독립 | 뷰별 독립 인코더 권장 |

**특히 주목할 특징**: brightness r이 높으면 조명이 균일, laplacian r이 낮으면 뷰마다 초점 거리가 달라  
선명도 정보가 다를 수 있음 → 이 경우 blur augmentation의 강도를 뷰별로 달리 설정하세요.


In [ ]:

# ── 5-2: Cross-view feature correlation ──────────────────────────────────────
# High correlation → views carry similar info → shared encoder assumption holds

# Pivot to wide: one row per (session, step)
cv_wide_rows = []
for sess_dir in ALL_SESSIONS[:12]:
    info = parse_session(sess_dir)
    with open(sess_dir / 'steps.jsonl') as f:
        lines = f.readlines()
    ins = [json.loads(l) for l in lines if json.loads(l)['phase']=='insert']
    for d in ins[::max(1,len(ins)//10)][:10]:
        obs = d.get('observation', {})
        row = {'session': info['session'], 'step': d['step']}
        all_ok = True
        for view in VIEWS:
            k = f'{view}_image'
            if k not in obs: all_ok = False; break
            try:
                rgb  = load_rgb(Path(obs[k]['path']))
                gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
                hsv  = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
                row[f'{view}_brightness'] = float(hsv[:,:,2].mean()/255)
                row[f'{view}_laplacian']  = float(cv2.Laplacian(gray,cv2.CV_32F).var())
                row[f'{view}_fft']        = fft_high_ratio(gray.astype(np.uint8))
                row[f'{view}_edge']       = float((cv2.Canny(gray.astype(np.uint8),40,120)>0).mean())
            except: all_ok = False; break
        if all_ok: cv_wide_rows.append(row)

wide_df = pd.DataFrame(cv_wide_rows)

feat_names = ['brightness','laplacian','fft','edge']
fig, axes = plt.subplots(1, len(feat_names), figsize=(18, 4))
corr_out = {}
for ax, feat in zip(axes, feat_names):
    lc = f'left_{feat}'; cc = f'center_{feat}'; rc = f'right_{feat}'
    if not all(c in wide_df for c in [lc,cc,rc]): continue
    r_lc = wide_df[lc].corr(wide_df[cc])
    r_lr = wide_df[lc].corr(wide_df[rc])
    r_cr = wide_df[cc].corr(wide_df[rc])
    corr_out[feat] = {'L-C': r_lc, 'L-R': r_lr, 'C-R': r_cr}
    ax.scatter(wide_df[lc], wide_df[cc], alpha=0.5, s=8, label=f'L-C r={r_lc:.2f}')
    ax.scatter(wide_df[cc], wide_df[rc], alpha=0.5, s=8, marker='s', label=f'C-R r={r_cr:.2f}')
    ax.set_title(feat); ax.legend(fontsize=7)

plt.suptitle('Cross-view Feature Correlation  [Shared Encoder Validity]', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nPearson r (r>0.8 = views carry similar info → shared encoder OK):')
display(pd.DataFrame(corr_out).T.round(3))


## Section 6 — Bias Summary (설계 결론)

### 무엇을 보는가
Section 1~5에서 발견한 편향들을 하나의 표로 정리하고,  
백본 인코더 학습 전 반드시 적용해야 할 조치를 우선순위 순으로 출력합니다.

### 어떻게 판단하는가
**우선순위 결정 기준**

| 편향 종류 | 영향 범위 | 조치 긴급도 |
|-----------|----------|-------------|
| Phase 불균형 (input) | 모든 학습 배치 | 🔴 필수 — 오버샘플링 또는 가중치 샘플링 |
| Task 불균형 (input) | 모든 학습 배치 | 🔴 필수 — 배치 내 1:1 샘플링 |
| Tip error bias (output) | 정책 일반화 | 🟡 중요 — 삽입 시작점 다양화 |
| 공간 편향 BIASED (input) | 인코더 ROI | 🟡 중요 — RandomCrop / Affine |
| 공간 편향 caution (input) | 인코더 ROI | 🟢 권장 — 약한 augmentation |

> **최종 체크리스트**: 이 셀의 RECOMMENDATIONS 항목이 모두 학습 코드에 반영되었는지  
> 학습 시작 전 확인하세요. 이 분석에서 발견하지 못한 편향은 Section 7~9에서 보완됩니다.


In [ ]:

# ── 6: Bias Summary ──────────────────────────────────────────────────────────
phase_cnt = steps_df['phase'].value_counts()
task_cnt  = steps_df['task_type'].value_counts()
ins = steps_df[steps_df['phase']=='insert']

print('='*65)
print('  Bias Summary')
print('='*65)

print('\n[1] PHASE IMBALANCE  (input frame distribution)')
for ph in ['approach','insert','stabilize']:
    c = phase_cnt.get(ph, 0)
    print(f'    {ph:12s}: {c:6d}  ({c/phase_cnt.sum()*100:.1f}%)')
print('    -> oversample approach or subsample insert uniformly')

print('\n[2] TASK TYPE IMBALANCE')
for t, c in task_cnt.items():
    print(f'    {t:6s}: {c:6d}  ({c/task_cnt.sum()*100:.1f}%)')
print('    -> balanced batch sampling 1:1')

print('\n[3] TIP ERROR  (output distribution)')
for task in ['nic','sc']:
    sub = ins[ins['task_type']==task]
    xe, ye = sub['tip_x_error'].mean()*1000, sub['tip_y_error'].mean()*1000
    xs, ys = sub['tip_x_error'].std()*1000,  sub['tip_y_error'].std()*1000
    sym = 'BIASED' if abs(xe)>xs*0.3 or abs(ye)>ys*0.3 else 'symmetric'
    print(f'    {task.upper()}: x={xe:+.2f}mm (std={xs:.2f})  y={ye:+.2f}mm (std={ys:.2f})  [{sym}]')
print('    -> add Gaussian offset at insert start to diversify error range')

print('\n[4] SPATIAL INPUT BIAS  (see variance heatmap)')
for (view, task), vm in sorted(var_maps.items()):
    c = spatial_conc(vm)
    flag = 'BIASED' if c < 0.15 else ('OK' if c > 0.30 else 'caution')
    print(f'    {task:4s} {view:7s}: {c:.3f}  [{flag}]')
print('    -> RandomCrop / Affine augmentation if BIASED')

print('\n[5] RECOMMENDATIONS')
recs = [
    '1. Oversample approach ~4x relative to insert',
    '2. Batch sample nic:sc = 1:1',
    '3. Add Gaussian XY offset (sigma~5mm) at insert start for diverse tip_error',
    '4. For spatially biased views: RandomCrop + AffineTransform augmentation',
    '5. Temporal shuffle within session to prevent z_offset overfitting',
]
for r in recs:
    print(f'   {r}')


## Section 7 — Temporal Redundancy (프레임 간 시간적 중복성)

### 무엇을 보는가
연속된 두 프레임이 얼마나 '다른가'를 시간 축으로 측정합니다.  
구체적으로는 mean absolute pixel difference (MAD)와 SSIM(Structural Similarity)을 
step 단위로 계산해 분포를 시각화합니다.

### 어떻게 판단하는가
| MAD 평균 | 의미 | 대응 |
|----------|------|------|
| < 1.5 (0~255 scale) | 연속 프레임이 사실상 동일 — 10Hz 샘플링에도 중복 심각 | 학습 시 temporal stride 3~5 이상 권장 |
| 1.5 ~ 5 | 적당한 변화량 — 그대로 써도 무방 | stride 2 정도 |
| > 5 | 프레임 간 변화가 크다 — 모션이 빠르거나 policy 전환 구간 | stride 1, 모든 프레임 활용 |

**핵심 질문 두 가지:**  
1. **approach vs insert 위상별 MAD 차이가 있는가?**  
   approach는 로봇이 빠르게 이동 → MAD가 높아야 정상.  
   insert는 미세 조정 → MAD가 낮아도 정상.  
   만약 두 위상이 비슷하다면 위상 구분이 시각적으로 어렵다는 신호.

2. **미니배치 내 유사 프레임 비율이 높은가?**  
   SSIM > 0.95인 연속 프레임 비율이 30% 이상이면 배치 샘플링 시  
   temporal stride 또는 random interval sampling 전략이 필요합니다.


In [ ]:
# ── 7-1: Frame-to-frame temporal redundancy ──────────────────────────────────
# Metric: MAD (mean absolute pixel diff) between consecutive steps
# High SSIM / low MAD → many nearly-identical frames → encoder sees same input repeatedly

from skimage.metrics import structural_similarity as ssim_fn

N_SESSIONS_TEMP = min(10, len(ALL_SESSIONS))
RESIZE_TEMP_H, RESIZE_TEMP_W = 128, 144   # smaller for speed

temp_rows = []
for sess_dir in tqdm(ALL_SESSIONS[:N_SESSIONS_TEMP], desc='Temporal redundancy'):
    info = parse_session(sess_dir)
    with open(sess_dir / 'steps.jsonl') as f:
        lines = f.readlines()

    prev_gray = None
    for ln in lines:
        d = json.loads(ln)
        obs = d.get('observation', {})
        key = 'center_image'
        if key not in obs:
            prev_gray = None
            continue
        try:
            img = cv2.imread(str(Path(obs[key]['path'])), cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, (RESIZE_TEMP_W, RESIZE_TEMP_H)).astype(np.float32)
        except Exception:
            prev_gray = None
            continue

        if prev_gray is not None:
            mad  = float(np.abs(img - prev_gray).mean())
            sv   = float(ssim_fn(img, prev_gray, data_range=255))
            temp_rows.append({
                'session':   info['session'],
                'task_type': info['task_type'],
                'phase':     d['phase'],
                'step':      d['step'],
                'mad':       mad,
                'ssim':      sv,
            })
        prev_gray = img

temp_df = pd.DataFrame(temp_rows)
print(f'Consecutive frame pairs: {len(temp_df):,}')

# ── 7-2: Visualise ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# MAD distribution by phase
sns.histplot(data=temp_df, x='mad', hue='phase', bins=40,
             stat='density', common_norm=False, element='step',
             fill=False, ax=axes[0])
axes[0].axvline(1.5, color='red', ls='--', lw=1, label='low-change threshold')
axes[0].set_title('MAD distribution by phase')
axes[0].set_xlabel('Mean Absolute Pixel Difference (0-255)')
axes[0].legend()

# SSIM distribution by phase
sns.histplot(data=temp_df, x='ssim', hue='phase', bins=40,
             stat='density', common_norm=False, element='step',
             fill=False, ax=axes[1])
axes[1].axvline(0.95, color='red', ls='--', lw=1, label='near-identical threshold')
axes[1].set_title('SSIM distribution by phase')
axes[1].set_xlabel('SSIM (1 = identical)')
axes[1].legend()

# MAD time-series for a few sessions
for sess, sg in temp_df.groupby('session'):
    axes[2].plot(sg['step'], sg['mad'], alpha=0.4, lw=0.7)
axes[2].set_title('MAD over steps (per session)')
axes[2].set_xlabel('Step')
axes[2].set_ylabel('MAD')

plt.suptitle('Temporal Redundancy  [Frame-to-Frame Change]', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# ── 7-3: Summary stats ───────────────────────────────────────────────────────
high_ssim_ratio = (temp_df['ssim'] > 0.95).mean()
print(f'\nFraction of consecutive pairs with SSIM > 0.95 : {high_ssim_ratio:.2%}')
print(f'  → {"REDUNDANT — use temporal stride >= 3" if high_ssim_ratio > 0.30 else "OK — stride 1~2 sufficient"}')
print()
print('MAD stats by phase:')
display(temp_df.groupby('phase')['mad'].describe().round(3))


## Section 8 — PCA / t-SNE Separability (시각적 분리 가능성)

### 무엇을 보는가
이미지 픽셀을 PCA로 차원 축소한 뒤 t-SNE로 2D 시각화합니다.  
각 점에 **task_type(NIC/SC)** 과 **phase(approach/insert)** 라벨을 색으로 표시합니다.

### 어떻게 판단하는가
**t-SNE 결과가 말해주는 것:**

| 패턴 | 의미 | 대응 |
|------|------|------|
| NIC / SC 클러스터가 명확히 분리 | 태스크 구분 정보가 이미지에 충분히 존재 | 인코더가 task-type feature를 학습할 수 있음 |
| NIC / SC가 완전히 섞임 | 두 태스크가 시각적으로 구별 불가 | 시각 정보 외 추가 모달리티 필요 |
| phase 라벨이 분리 | approach↔insert가 시각적으로 다름 | phase-conditional 인코더 고려 |
| phase 라벨이 섞임 | approach와 insert가 시각적으로 비슷 | phase를 visual signal로 쓰기 어려움 |

**PCA explained variance:**  
- 상위 10 PC가 분산의 70% 이상 설명 → 이미지 정보가 저차원에 집중 (좋음)  
- 50 PC를 써도 50% 미만 → 이미지가 복잡하거나 노이즈가 많음 → 더 큰 인코더 필요

> **주의:** t-SNE는 시각화 도구일 뿐, 클러스터 크기/거리는 정량적 의미가 없습니다.  
> 클러스터의 **유무**와 **라벨 순도**만 판단하세요.


In [ ]:
# ── 8-1: PCA / t-SNE separability ───────────────────────────────────────────
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

RESIZE_PCA_H, RESIZE_PCA_W = 64, 72   # small for PCA
N_PER_SESS_PCA = 8
N_PCA_COMPONENTS = 50

pca_rows, pca_imgs = [], []

for sess_dir in tqdm(ALL_SESSIONS, desc='Loading images for PCA'):
    info = parse_session(sess_dir)
    with open(sess_dir / 'steps.jsonl') as f:
        lines = f.readlines()
    idxs = np.linspace(0, len(lines)-1, N_PER_SESS_PCA, dtype=int)
    for i in idxs:
        d = json.loads(lines[i])
        obs = d.get('observation', {})
        key = 'center_image'
        if key not in obs:
            continue
        try:
            img = cv2.imread(str(Path(obs[key]['path'])), cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, (RESIZE_PCA_W, RESIZE_PCA_H)).astype(np.float32) / 255.0
            pca_imgs.append(img.flatten())
            pca_rows.append({
                'session':   info['session'],
                'task_type': info['task_type'],
                'phase':     d['phase'],
            })
        except Exception:
            continue

X = np.array(pca_imgs)
meta_df = pd.DataFrame(pca_rows)
print(f'Image matrix: {X.shape}  ({X.shape[0]} frames × {X.shape[1]} pixels)')

# PCA
n_components = min(N_PCA_COMPONENTS, X.shape[0] - 1)
pca = PCA(n_components=n_components, random_state=42)
X_pca = pca.fit_transform(X)

# t-SNE on PCA embedding
tsne = TSNE(n_components=2, perplexity=min(30, len(X_pca)//4),
            random_state=42, n_iter=800, verbose=0)
X_2d = tsne.fit_transform(X_pca)

# ── 8-2: Plot ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) PCA explained variance
ev = np.cumsum(pca.explained_variance_ratio_) * 100
axes[0].plot(range(1, len(ev)+1), ev, marker='o', ms=3)
axes[0].axhline(70, color='red', ls='--', lw=1, label='70% threshold')
axes[0].axhline(90, color='orange', ls='--', lw=1, label='90% threshold')
for n, thr in [(10, 70), (20, 80)]:
    if n <= len(ev):
        axes[0].annotate(f'{ev[n-1]:.0f}% @ PC{n}',
                         xy=(n, ev[n-1]), xytext=(n+2, ev[n-1]-8), fontsize=8,
                         arrowprops=dict(arrowstyle='->', lw=0.8))
axes[0].set_xlabel('Number of PCs')
axes[0].set_ylabel('Cumulative explained variance (%)')
axes[0].set_title('PCA Explained Variance')
axes[0].legend(fontsize=8)

# (b) t-SNE coloured by task_type
palette = {'nic': '#2196F3', 'sc': '#FF5722'}
for task, color in palette.items():
    mask = meta_df['task_type'] == task
    axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1],
                    c=color, alpha=0.5, s=8, label=task)
axes[1].set_title('t-SNE  coloured by task_type')
axes[1].legend(markerscale=3)
axes[1].axis('off')

# (c) t-SNE coloured by phase
phase_palette = {'approach': '#4CAF50', 'insert': '#9C27B0', 'stabilize': '#FF9800'}
for phase, color in phase_palette.items():
    mask = meta_df['phase'] == phase
    if mask.sum() == 0: continue
    axes[2].scatter(X_2d[mask, 0], X_2d[mask, 1],
                    c=color, alpha=0.5, s=8, label=phase)
axes[2].set_title('t-SNE  coloured by phase')
axes[2].legend(markerscale=3)
axes[2].axis('off')

plt.suptitle('PCA / t-SNE Separability  [Encoder Learnability]', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# ── 8-3: Linear separability score (logistic regression on PCA features) ─────
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

for label_col, label_name in [('task_type', 'NIC vs SC'), ('phase', 'phase')]:
    y = meta_df[label_col].values
    valid = ~pd.Series(y).isna()
    score = cross_val_score(
        LogisticRegression(max_iter=300, random_state=42),
        X_pca[valid], y[valid], cv=5, scoring='accuracy'
    ).mean()
    baseline = pd.Series(y[valid]).value_counts(normalize=True).max()
    delta = score - baseline
    flag = 'SEPARABLE' if delta > 0.10 else ('marginal' if delta > 0.03 else 'NOT separable')
    print(f'[{label_name:12s}] linear acc={score:.3f}  baseline={baseline:.3f}  '
          f'delta={delta:+.3f}  [{flag}]')


## Section 9 — Inter-session Diversity (세션 간 시각적 다양성)

### 무엇을 보는가
**같은 rail·task 조건**에서 수집된 서로 다른 세션들의 평균 이미지를 비교합니다.  
세션 간 brightness/hue 히스토그램 거리와 mean-image SSIM을 측정합니다.

### 어떻게 판단하는가
**인코더 overfitting 위험 지표:**

| 지표 | 임계값 | 의미 |
|------|--------|------|
| 세션 간 mean-image SSIM | > 0.90 | 모든 세션이 거의 같은 장면 → 인코더가 배경을 암기할 위험 |
| 세션 간 brightness std | < 0.02 | 조명 변화 없음 → 조명 다양성 augmentation 필수 |
| 세션 간 hue histogram L1 | < 0.05 | 색상 분포가 고정 → ColorJitter augmentation 필수 |

**핵심 해석 방법:**  
- Mean image가 세션마다 거의 동일하게 보이면 → 배경이 고정, 인코더가 배경 특징에 의존하게 됨  
- 세션 간 SSIM이 낮으면(< 0.7) → 수집 조건(조명, 물체 위치)에 이미 충분한 다양성  
- Rail별 다양성을 따로 보는 이유: 같은 rail은 카메라 앵글이 유사 → 더 취약

**결론으로 연결:**  
세션 간 다양성이 낮을수록 augmentation 강도를 높여야 하며,  
특히 ColorJitter, RandomBrightness, RandomAffine 등 외관 변화 aug가 효과적입니다.


In [ ]:
# ── 9-1: Inter-session diversity ─────────────────────────────────────────────
# For each (task_type, rail) group: compute mean image per session → compare

from skimage.metrics import structural_similarity as ssim_fn

RESIZE_DIV_H, RESIZE_DIV_W = 128, 144
N_FRAMES_MEAN = 20  # frames per session for mean image

sess_means = {}   # (session_name) → mean_gray image
sess_meta  = {}   # (session_name) → {task_type, rail}

for sess_dir in tqdm(ALL_SESSIONS, desc='Computing session mean images'):
    info = parse_session(sess_dir)
    with open(sess_dir / 'steps.jsonl') as f:
        lines = f.readlines()
    idxs = np.linspace(0, len(lines)-1, N_FRAMES_MEAN, dtype=int)
    frames = []
    for i in idxs:
        d = json.loads(lines[i])
        obs = d.get('observation', {})
        key = 'center_image'
        if key not in obs: continue
        try:
            img = cv2.imread(str(Path(obs[key]['path'])), cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, (RESIZE_DIV_W, RESIZE_DIV_H)).astype(np.float32)
            frames.append(img)
        except Exception:
            continue
    if len(frames) >= 5:
        sess_means[info['session']] = np.mean(frames, axis=0)
        sess_meta[info['session']]  = {'task_type': info['task_type'], 'rail': info['rail']}

print(f'Sessions with mean image: {len(sess_means)}')

# ── 9-2: Pairwise SSIM within same (task_type, rail) group ───────────────────
from itertools import combinations

div_rows = []
groups = {}
for sname, meta in sess_meta.items():
    key = (meta['task_type'], meta['rail'])
    groups.setdefault(key, []).append(sname)

for group_key, sessions in groups.items():
    if len(sessions) < 2: continue
    for s1, s2 in combinations(sessions, 2):
        sim = ssim_fn(sess_means[s1], sess_means[s2], data_range=255)
        div_rows.append({
            'task_type': group_key[0], 'rail': group_key[1],
            'group': f"{group_key[0]}_{group_key[1]}",
            's1': s1, 's2': s2, 'ssim': sim
        })

div_df = pd.DataFrame(div_rows)

# ── 9-3: Visualise ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# (a) SSIM distribution
if len(div_df):
    sns.boxplot(data=div_df, x='group', y='ssim', ax=axes[0], showfliers=False)
    axes[0].axhline(0.90, color='red', ls='--', lw=1, label='overfitting risk')
    axes[0].axhline(0.70, color='green', ls='--', lw=1, label='good diversity')
    axes[0].set_title('Inter-session SSIM (same rail/task group)')
    axes[0].set_ylabel('SSIM (higher = more similar = less diverse)')
    axes[0].tick_params(axis='x', labelrotation=30, labelsize=8)
    axes[0].legend(fontsize=8)

# (b) Mean images side-by-side: pick first group with >= 2 sessions
sample_sessions = list(sess_means.keys())[:4]
for ax, sname in zip(axes[1:], sample_sessions[:2]):
    ax.imshow(sess_means[sname], cmap='gray', vmin=0, vmax=255)
    m = sess_meta[sname]
    ax.set_title(f"mean image\n{m['task_type']} {m['rail']}", fontsize=9)
    ax.axis('off')

plt.suptitle('Inter-session Diversity  [Overfitting Risk]', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# ── 9-4: Brightness & hue spread ─────────────────────────────────────────────
bri_rows = []
for sess_dir in ALL_SESSIONS:
    info = parse_session(sess_dir)
    with open(sess_dir / 'steps.jsonl') as f:
        lines = f.readlines()
    vals = []
    for ln in lines[::max(1, len(lines)//10)][:10]:
        d = json.loads(ln)
        obs = d.get('observation', {})
        if 'center_image' not in obs: continue
        try:
            rgb = cv2.imread(str(Path(obs['center_image']['path'])), cv2.IMREAD_COLOR)
            hsv = cv2.cvtColor(rgb, cv2.COLOR_BGR2HSV)
            vals.append(hsv[:,:,2].mean() / 255.0)
        except: continue
    if vals:
        bri_rows.append({'session': info['session'],
                         'task_type': info['task_type'],
                         'mean_brightness': float(np.mean(vals))})

bri_df = pd.DataFrame(bri_rows)

# ── 9-5: Summary ─────────────────────────────────────────────────────────────
print('\n=== Inter-session Diversity Summary ===')
if len(div_df):
    mean_ssim = div_df['ssim'].mean()
    flag = 'RISK: low diversity' if mean_ssim > 0.90 else ('OK' if mean_ssim < 0.75 else 'caution')
    print(f'  Mean inter-session SSIM (same group): {mean_ssim:.3f}  [{flag}]')

bri_std = bri_df['mean_brightness'].std()
flag_b = 'RISK: no lighting variation' if bri_std < 0.02 else ('OK' if bri_std > 0.05 else 'low')
print(f'  Brightness std across sessions      : {bri_std:.4f}  [{flag_b}]')
print()
print('Recommended augmentation based on diversity:')
if bri_std < 0.05:
    print('  → RandomBrightness / ColorJitter (no lighting variation in data)')
if len(div_df) and div_df['ssim'].mean() > 0.80:
    print('  → RandomAffine / RandomCrop (scene is too static across sessions)')
